## Setup & Rubric Compliant Paths

In [ ]:
import pandas as pd
import requests
import time
import os
from pathlib import Path

print("Starting Phase 1: Data Ingestion Prototype...")

# RUBRIC COMPLIANCE: Use pathlib.Path, no hardcoded strings!
BASE_DIR = Path('..').resolve()
RAW_DIR = BASE_DIR / 'data' / 'raw'

# Ensure the raw directory exists
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Target directory verified at: {RAW_DIR}")

## The API Extraction Function

In [ ]:
def fetch_nav_data(amfi_code):
    """
    Fetches historical NAV data from mfapi.in for a given AMFI code.
    Includes built-in error handling for network timeouts or bad responses.
    """
    url = f"https://api.mfapi.in/mf/{amfi_code}"
    
    try:
        response = requests.get(url, timeout=10)
        # Raise an error if the HTTP request returned an unsuccessful status code
        response.raise_for_status() 
        
        data = response.json()
        
        # Check if the API returned actual data
        if 'data' not in data or len(data['data']) == 0:
            print(f"⚠️ Warning: No data found for AMFI code {amfi_code}")
            return None
            
        # Convert the JSON payload into a Pandas DataFrame
        df = pd.DataFrame(data['data'])
        df['amfi_code'] = amfi_code # Tag the data with the fund code
        df['scheme_name'] = data['meta']['scheme_name']
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Network Error fetching {amfi_code}: {e}")
        return None
    except Exception as e:
        print(f"❌ Data Parsing Error for {amfi_code}: {e}")
        return None

print("✅ Fetch function defined with full error handling.")

## Executing the Extraction (The 40 Schemes)

In [ ]:
print("Initiating API calls to mfapi.in...")

# Sample list of AMFI codes (e.g., SBI Bluechip, HDFC Top 100, Parag Parikh Flexi)
# You will expand this list to your full 40 schemes
target_schemes = [120716, 119062, 122639, 118287, 120503] 

all_nav_data = []

for code in target_schemes:
    print(f"Fetching data for AMFI Code: {code}...")
    df_fund = fetch_nav_data(code)
    
    if df_fund is not None:
        all_nav_data.append(df_fund)
        print(f"   -> Success! Fetched {len(df_fund)} rows.")
    
    # CRITICAL: Sleep for 1 second between API calls so we don't get blocked by the server
    time.sleep(1) 

# Combine all individual fund dataframes into one master dataframe
if all_nav_data:
    df_master_nav = pd.concat(all_nav_data, ignore_index=True)
    print(f"\n✅ All data fetched! Total records: {len(df_master_nav)}")
else:
    print("\n❌ Failed to fetch any data.")

## Saving to the Raw Output Folder

In [ ]:
# Rename columns to match our Day 2 cleaning scripts perfectly
df_master_nav = df_master_nav.rename(columns={'date': 'date', 'nav': 'nav'})

# Save the raw file
output_file = RAW_DIR / '02_nav_history.csv'
df_master_nav.to_csv(output_file, index=False)

print(f"✅ Raw NAV data successfully saved to: {output_file}")
df_master_nav.head()

## Documentation & Next Steps (Markdown)

### 🚀 Moving to Production (Deliverable D1 & Bonus B1)
This notebook successfully prototypes the extraction logic. To achieve full marks for **D1 (ETL Pipeline)** and the **+10 Bonus (B1)**, this code must be translated into the `scripts/etl_pipeline.py` file.

**Production Setup Instructions:**
1. Copy the `fetch_nav_data()` function into `etl_pipeline.py`.
2. Add a `logging` module to write errors to a `pipeline.log` file instead of printing them.
3. Configure a Cron Job (Linux/Mac) or Task Scheduler (Windows) to execute the script:
   * **Cron Expression for Weekdays at 8 PM:** `0 20 * * 1-5 /usr/bin/python3 /path/to/bluestock_mf_capstone/scripts/etl_pipeline.py`